![Imgur](https://i.imgur.com/acSOZRh.png)

# Laboratorio n° 1. Parte A: Fundamentos de PyTorch

**Asignatura:** Redes Neuronales Profundas
**Bloque:** 1 — Fundamentos de Deep Learning

---

## Introducción

PyTorch es el framework de deep learning más utilizado en investigación y se está volviendo dominante también en la industria. Toda red neuronal en PyTorch —sin importar su complejidad— se construye sobre un único concepto fundamental: el **tensor**.

Este trabajo práctico tiene como objetivo que te familiarices con las operaciones sobre tensores que más se usan en el día a día del deep learning:

- Crear y explorar tensores
- Cambiar su forma en memoria (`reshape`, `view`, `unsqueeze`)
- Entender cómo PyTorch opera tensores de distintas formas (*broadcasting*)
- Calcular derivadas automáticamente con `autograd`

---

## Instrucciones generales

- Completá el código en las celdas marcadas con `# Tu código aquí`.
- Respondé las preguntas de análisis en las celdas de texto (tipo Markdown).
- Para resolver cada ejercicio, consultá el material teórico de la Clase 1.
- **No está permitido usar bucles `for` o `while` salvo que el enunciado lo indique explícitamente.**

In [1]:
import torch
print(f"Versión de PyTorch: {torch.__version__}")

Versión de PyTorch: 2.10.0+cpu


---
## Sección A: Creación y atributos de tensores

### Ejercicio 1 — Creación y exploración de un tensor tridimensional

**Objetivo:** Entender cómo se estructura la información en tensores de varias dimensiones y qué atributos los describen.

**Enunciado:**

1. Inicializá un tensor tridimensional lleno de **ceros** con forma `(2, 3, 4)` — es decir, 2 bloques, 3 filas y 4 columnas. Utilizar la función de PyTorch diseñada para esto.
2. A partir de ese tensor, creá otro tensor con **valores aleatorios** (entre 0 y 1) que tenga exactamente la misma forma, usando la función `*_like` correspondiente.
3. Imprimí por pantalla los siguientes atributos del tensor aleatorio:
- Su forma (`.shape`)
- Su tipo de dato (`.dtype`)
- Su número de dimensiones (`.ndim`)
- La cantidad total de elementos (`.numel()`)



In [17]:
# Tu código aquí
tensor_ceros = torch.zeros(2, 3, 4)

tensor_random = torch.rand_like(tensor_ceros)
print(tensor_ceros)
print(tensor_random)

print(tensor_ceros.size())
print(tensor_random.size())
print(tensor_random.shape)
print(tensor_random.dtype)
print(tensor_random.ndim)
print(tensor_random.numel())


tensor([[[0., 0., 0., 0.],
         [0., 0., 0., 0.],
         [0., 0., 0., 0.]],

        [[0., 0., 0., 0.],
         [0., 0., 0., 0.],
         [0., 0., 0., 0.]]])
tensor([[[0.8551, 0.7384, 0.5975, 0.5453],
         [0.8480, 0.6931, 0.0460, 0.9717],
         [0.7642, 0.4240, 0.8459, 0.7787]],

        [[0.8226, 0.7108, 0.2165, 0.4732],
         [0.1587, 0.0064, 0.3059, 0.6802],
         [0.4278, 0.3005, 0.4963, 0.8852]]])
torch.Size([2, 3, 4])
torch.Size([2, 3, 4])
torch.Size([2, 3, 4])
torch.float32
3
24


**Pregunta de análisis:**

¿Qué diferencia hay entre `torch.zeros(2, 3, 4)` y `torch.empty(2, 3, 4)`? ¿En qué situación usarías cada uno?

el primero crea un tensor de estas dimensiones (2, 3, 4) lleno de ceros y el segundo crea un vector de estas dimensiones (2, 3, 4) pero los valores de cada celda son los que ya estaban ocupando el lugar que vamos a utilizar, es decir pueden ser cualquier tipo de dato o número

### Ejercicio 2 — Tipos de dato y conversión

**Objetivo:** Comprender que los tensores tienen un tipo de dato numérico subyacente y que elegirlo bien tiene consecuencias en precisión y rendimiento.

**Enunciado:**

1. Creá un tensor de números aleatorios con forma `(3, 3)` con el tipo de dato por defecto (`float32`).
2. Convertilo a `float16`.
3. Imprimí ambos tensores y sus respectivos `dtype`.
4. Volvé a convertir el tensor de `float16` nuevamente a `float32`. Luego, calculá la diferencia (resta) entre el tensor original y este último. Observá si los valores de la resta son exactamente cero o si hay pequeñas diferencias.


In [25]:
tensor_random_float32 = torch.rand(3,3,dtype=torch.float32)
print(tensor_random_float32)
print(tensor_random_float32.dtype)
tensor_random_float16 = tensor_random_float32.type(torch.float16)
print(tensor_random_float16)
print(tensor_random_float16.dtype)
tensor_random_float32_2 = tensor_random_float16.type(torch.float32)
print(tensor_random_float32 - tensor_random_float32_2)
print(tensor_random_float32.dtype)

tensor([[0.7534, 0.9087, 0.8251],
        [0.3363, 0.6447, 0.5958],
        [0.7892, 0.4530, 0.3639]])
torch.float32
tensor([[0.7534, 0.9087, 0.8252],
        [0.3364, 0.6445, 0.5957],
        [0.7891, 0.4531, 0.3640]], dtype=torch.float16)
torch.float16
tensor([[ 2.5034e-06, -1.6689e-06, -7.4387e-05],
        [-8.2016e-05,  1.4126e-04,  1.1259e-04],
        [ 1.6856e-04, -9.9659e-05, -1.1003e-04]])
torch.float32


**Pregunta de análisis:**

`float16` usa la mitad de memoria que `float32`. ¿Por qué, aun con esa ventaja, no siempre es conveniente entrenar redes en `float16`? ¿Qué riesgo implica la menor precisión numérica?

Aunque float16 usa menos memoria, no siempre conviene para entrenar porque tiene menor precisión y rango numérico, lo que puede hacer que gradientes muy pequeños se vuelvan cero (underflow) o que valores grandes exploten a infinito o NaN (overflow); esto distorsiona la información de los gradientes y puede volver el entrenamiento inestable o incluso hacer que el modelo deje de aprender correctamente.

---
## Sección B: Cambio de forma — `reshape` y `view`

### Ejercicio 3 — Reorganización bidimensional y orden de llenado

**Objetivo:** Analizar en qué orden PyTorch acomoda los elementos al cambiar la forma de un tensor.

**Enunciado:**

1. Creá un tensor unidimensional con los números del 1 al 12.
2. Reorganizálo en una matriz de **3 filas × 4 columnas** usando `reshape()`.
3. Imprimí la matriz e identificá en qué fila y columna quedó el número `8`.
4. Ahora reorganizá el mismo vector original en una matriz de **4 filas × 3 columnas** y comparálo con el resultado anterior.

> **Funciones a utilizar:** `torch.arange()` y `.reshape()`.

In [30]:
# Tu código aquí
tensor_unidimensional = torch.arange(1,13)
print(tensor_unidimensional)

tensor_unidimensional_reorganizado = tensor_unidimensional.reshape(3,4)
print(tensor_unidimensional_reorganizado)

tensor_unidimensional_reorganizado_2 = tensor_unidimensional.reshape(4, 3)
print(tensor_unidimensional_reorganizado_2)

tensor_unidimensional_reconstruido = tensor_unidimensional_reorganizado_2.flatten()
print(tensor_unidimensional_reconstruido)


tensor([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12])
tensor([[ 1,  2,  3,  4],
        [ 5,  6,  7,  8],
        [ 9, 10, 11, 12]])
tensor([[ 1,  2,  3],
        [ 4,  5,  6],
        [ 7,  8,  9],
        [10, 11, 12]])
tensor([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12])


**Pregunta de análisis:**

¿En qué orden llena PyTorch los elementos al hacer un `reshape`? ¿Cómo se denomina esa convención? (Pista: es la misma que usa NumPy y Python por defecto.)

Fila por fila

### Ejercicio 4 — Orden de llenado en cuatro dimensiones (Tensores 4D)

**Objetivo:** Extender el concepto del orden de llenado a tensores de 4 dimensiones, como los frecuentemente usados para lotes de imágenes: `(lotes, canales, filas, columnas)`.

**Enunciado:**

1. Creá un tensor unidimensional con los números del 1 al 24.
2. Reorganizálo en un tensor 4D con forma `(2, 3, 2, 2)`. Es decir: 2 lotes (batches), 3 canales, 2 filas y 2 columnas.
3. Imprimí el tensor e identificá el patrón de llenado. ¿Qué dimensiones se completan primero y cuáles últimas?
4. Extraé el valor almacenado en la posición `[lote 1, canal 2, fila 0, columna 1]` (recordando que los índices empiezan en 0) usando indexación, por ejemplo: `tensor[1, 2, 0, 1]`.
5. Verificá analíticamente si el valor extraído coincide con lo esperado.


In [39]:
# Tu código aquí
tensor_1_al_24 = torch.arange(1,25)
print(tensor_1_al_24)

tensor_4d = tensor_1_al_24.reshape(2,3,2,2)
print(tensor_4d)

# Se va llenando de adentro(izquierda) para afuera(derecha)

print(tensor_4d[1,2,0,1])

print(tensor_1_al_24[12+6+1+2])

tensor([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18,
        19, 20, 21, 22, 23, 24])
tensor([[[[ 1,  2],
          [ 3,  4]],

         [[ 5,  6],
          [ 7,  8]],

         [[ 9, 10],
          [11, 12]]],


        [[[13, 14],
          [15, 16]],

         [[17, 18],
          [19, 20]],

         [[21, 22],
          [23, 24]]]])
tensor(22)
tensor(22)


**Pregunta de análisis:**

En un tensor de forma `(lotes, canales, filas, columnas)`, ¿qué dimensión varía más rápido en memoria (es decir, qué elementos son adyacentes) y cuál más lento?

La ultima dimension(columnas)

### Ejercicio 5 — `view` en un tensor tridimensional

**Objetivo:** Extender el concepto de reorganización a tres dimensiones y practicar la indexación de tensores.

**Enunciado:**

1. Creá un tensor unidimensional con los números del 0 al 23.
2. Usá el método `.view()` para darle la forma `(2, 3, 4)`.
3. Extraé e imprimí el **primer bloque** (índice 0 de la primera dimensión).
*¿Qué forma tiene ese bloque?*
4. Sin usar bucles, identificá en qué posición `[bloque, fila, columna]` quedó el número `15`.

> **Pista:** Podés indexar un tensor tridimensional con tres índices: `tensor[bloque, fila, columna]`. Para encontrar el 15, podés calcularlo a mano usando el orden de llenado que aprendiste en el ejercicio anterior.

In [52]:
# Tu código aquí
tensor_0_a_23 = torch.arange(0,24)
print(tensor_0_a_23)

tensor_0_a_23_view = tensor_0_a_23.view(2,3,4)
print(tensor_0_a_23_view)

print(tensor_0_a_23_view[0])
# Tiene forma de una matriz de 3x4

# 12 + 0 + 4
print(tensor_0_a_23_view[1,0,3])

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19, 20, 21, 22, 23])
tensor([[[ 0,  1,  2,  3],
         [ 4,  5,  6,  7],
         [ 8,  9, 10, 11]],

        [[12, 13, 14, 15],
         [16, 17, 18, 19],
         [20, 21, 22, 23]]])
tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])
tensor(15)


Ejercicio 6 — Memoria compartida y `.clone()`

**Objetivo:** Entender que `view` y `reshape` no copian los datos en memoria, sino que crean una *vista* sobre los mismos datos. Aprender a hacer una copia real con `.clone()`.

**Parte A — La vista comparte memoria:**

1. Utilizá el vector de 24 elementos del Ejercicio 5 y su versión con forma `(2, 3, 4)` (el tensor cúbico).
2. Modificá el **primer elemento** del vector original asignándole el valor `-99`.
3. Imprimí el tensor cúbico y observá si también cambió.

**Parte B — Copiar con `.clone()`:**

4. Creá una copia *real* del tensor cúbico usando `.clone()`.
5. Modificá ahora el segundo elemento del vector original (asignale `-999`).
6. Imprimí la copia: esta vez **no** debería reflejar el cambio.

**Pista:** Un tensor creado con `view` o `reshape` a partir de otro comparte la misma zona de memoria. Cualquier modificación al original se refleja en todas sus vistas — a menos que uses `.clone()` para crear una copia independiente.

In [53]:
# Tu código aquí — Parte A
tensor_0_a_23[0] = -99
print(tensor_0_a_23_view[0])

tensor([[-99,   1,   2,   3],
        [  4,   5,   6,   7],
        [  8,   9,  10,  11]])


In [57]:
# Tu código aquí — Parte B
tensor_cubico_clonado = tensor_0_a_23_view.clone()
tensor_0_a_23[2] = -999
print(tensor_0_a_23_view[0,0,2])
print(tensor_cubico_clonado[0,0,2])

tensor(-999)
tensor(2)


**Pregunta de análisis:**

¿Qué ventaja de rendimiento tiene el hecho de que `view` y `reshape` no copien datos? ¿En qué situación sería un problema que dos tensores compartan la misma memoria, y cómo lo solucionarías?

Tiene la ventaja de que funciona todo mucho mas rapido, pero pueden haber problemas si no se maneja cuidadosamente

---
## Sección C: Broadcasting

### Introducción al broadcasting

El *broadcasting* es la capacidad de PyTorch (heredada de NumPy) de operar matemáticamente sobre tensores de **distintas formas**, sin copiar datos en memoria. PyTorch compara las dimensiones de los dos tensores **de derecha a izquierda** y aplica las siguientes reglas para cada par de dimensiones:

| Situación | Resultado |
|---|---|
| Ambas dimensiones son iguales | Se opera normalmente |
| Una de las dimensiones es `1` | Se "expande" virtualmente |
| Una dimensión no existe | Se trata como si fuera `1` |
| Distintas y ninguna es `1` | Error |

Esto es fundamental en deep learning: por ejemplo, cuando se suma un vector de *biases* a toda una matriz de activaciones, se usa broadcasting.

### Ejercicio 7 — Broadcasting elemental

**Objetivo:** Observar la regla más simple de broadcasting: un vector sobre las filas de una matriz.

**Enunciado:**

1. Creá una matriz `A` de forma `(3, 4)` llena de unos.
2. Creá un vector `v` de 4 elementos con los valores `[10, 20, 30, 40]`.
3. Calculá el producto `A * v` e imprimí el resultado.
4. Verificá que las reglas de broadcasting aplican: anotá las formas de `A` y `v` de derecha a izquierda y verificá si son compatibles.


In [58]:
# Tu código aquí
A = torch.ones(3,4)
v = torch.tensor([10,20,30,40])
print(A*v)

# Aplica una dimension no existe, se expande virtualmente


tensor([[10., 20., 30., 40.],
        [10., 20., 30., 40.],
        [10., 20., 30., 40.]])


**Pregunta de análisis:**

Verificá la compatibilidad comparando de derecha a izquierda las dimensiones de `A` (forma `(3, 4)`) y `v` (forma `(4)`):

```
A: 3 4
v: 4 ← la dimensión izquierda no existe, se trata como 1
```

¿Por qué el resultado final tiene forma `(3, 4)`?

Porque v se expande vitualmente para ser una matriz de (1,4)

### Ejercicio 8 — Broadcasting con `reshape`/`unsqueeze` en tensores 3D

**Objetivo:** Adaptar dimensiones con `reshape` o `unsqueeze` para hacer compatible el broadcasting cuando las formas no coinciden naturalmente.

**Enunciado:**

Tenés un tensor `A` con forma `(3, 2, 4)` y un vector `v` con 6 elementos `[6, 5, 4, 3, 2, 1]`.

El objetivo es multiplicar `A * v` de manera que **cada fila de `A` se multiplique por el elemento correspondiente de `v`**:
- La fila 0 (del bloque 0) se multiplica por 6
- La fila 1 (del bloque 0) se multiplica por 5
- La fila 0 (del bloque 1) se multiplica por 4
- ... y así sucesivamente

Para lograr esto, necesitás reorganizar `v` para que PyTorch sepa cómo alinearlo con `A`.


**Pasos:**

1. Generá e imprimí `A` y `v` para entender qué forma tienen.
1. Reorganizá `v` para que sea compatible con `A` usando `reshape` o `unsqueeze`.
1. Realizá la multiplicación y verificá que cada fila fue multiplicada por el escalar correcto.

> **Pista:** El tensor `A` tiene 3 bloques de 2 filas cada uno (3 × 2 = 6 filas en total). Necesitás que `v` tenga una forma que se alinee con las dos primeras dimensiones de `A`, dejando una dimensión de tamaño 1 al final para que el broadcasting la expanda. Pensá qué forma de 3 dimensiones le darías a `v`.

In [66]:
# Tu código aquí
A = torch.ones(3,2,4)
v = torch.tensor([6,5,4,3,2,1])
print(A)
print(v)

v_reorganizado = v.reshape(3,2,1)
print(v_reorganizado)

print(A*v_reorganizado)

tensor([[[1., 1., 1., 1.],
         [1., 1., 1., 1.]],

        [[1., 1., 1., 1.],
         [1., 1., 1., 1.]],

        [[1., 1., 1., 1.],
         [1., 1., 1., 1.]]])
tensor([6, 5, 4, 3, 2, 1])
tensor([[[6],
         [5]],

        [[4],
         [3]],

        [[2],
         [1]]])
tensor([[[6., 6., 6., 6.],
         [5., 5., 5., 5.]],

        [[4., 4., 4., 4.],
         [3., 3., 3., 3.]],

        [[2., 2., 2., 2.],
         [1., 1., 1., 1.]]])


### Ejercicio 9 — Normalización de canales de imágenes

**Objetivo:** Aplicar broadcasting en un contexto completamente real: el pre-procesamiento de imágenes en deep learning.

**Contexto:** En visión por computadora, antes de entrenar una red con imágenes, se *normaliza* cada canal de color restándole su media histórica. Un lote de imágenes tiene forma `(N, C, H, W)` donde:
- `N` = cantidad de imágenes en el lote
- `C` = canales de color (3 para imágenes RGB: Rojo, Verde, Azul)
- `H` = alto de la imagen en píxeles
- `W` = ancho de la imagen en píxeles

**Enunciado:**

Tenés un lote de imágenes `X` con forma `(32, 3, 28, 28)`, esto es un lote de 32 imágenes de tres canales (RGB) de 28x28 píxeles. Utilizarás también un vector llamado `medias` de 3 componentes donde cada componente es la media de cada canal (RGB) del dataset completo.

Tu tarea es calcular la resta `X - medias` de forma que a todos los canales de color en **todas** las imágenes del lote se le reste la media correspondiente. Todo sin usar bucles.

1. Intentá directamente `X - medias` y observá el error que da PyTorch. Leélo con atención.
2. Reorganizá `medias` para que PyTorch pueda alinearlo con la dimensión de canales (dim 1) de `X`.
3. Calculá la resta correctamente y verificá que el resultado tiene forma `(32, 3, 28, 28)`.

> **Pista:** Las reglas de broadcasting van de **derecha a izquierda**. Si intentás `X - medias` directamente, PyTorch intentará alinear el `3` de `medias` con el `28` del ancho, lo que falla. Necesitás que `medias` tenga una forma con dimensiones de tamaño 1 en las posiciones que no corresponden al canal. Buscá `.view()` o `.unsqueeze()` para agregar dimensiones de tamaño 1.

In [83]:
import torch
torch.manual_seed(42)
X = torch.rand(32, 3, 28, 28)
medias = torch.tensor([0.5, 0.4, 0.3])

# Tu código aquí: reorganizá medias y calculá la resta correctamente
medias = medias.unsqueeze(1)
medias = medias.unsqueeze(1)
medias = medias.unsqueeze(0)
print(X-medias)

tensor([[[[ 3.8227e-01,  4.1500e-01, -1.1714e-01,  ..., -2.3051e-01,
           -1.4119e-01, -3.0064e-01],
          [ 4.7192e-02, -4.9384e-01,  4.5155e-01,  ...,  4.1027e-01,
            1.4402e-01,  2.0711e-01],
          [ 1.5813e-01, -8.6980e-03,  3.9130e-01,  ..., -3.4086e-01,
            2.6529e-01, -2.0210e-01],
          ...,
          [ 3.0288e-01, -2.3379e-01, -2.3860e-01,  ...,  1.6827e-01,
            1.7790e-01, -4.1630e-01],
          [-4.8501e-01, -2.5944e-01,  3.4227e-01,  ..., -6.9003e-03,
            4.5762e-01, -3.0011e-01],
          [ 3.9311e-03,  2.3780e-01, -3.4518e-01,  ..., -1.9818e-01,
            1.3013e-01,  1.8857e-01]],

         [[-1.6337e-01, -3.9579e-01,  3.6172e-01,  ..., -2.0540e-01,
           -1.4605e-01,  1.9613e-01],
          [ 2.3563e-01,  2.9224e-01,  3.7444e-01,  ...,  5.8341e-02,
            2.0788e-01, -1.7420e-01],
          [ 2.4424e-01, -3.8821e-01, -2.5775e-01,  ...,  5.1839e-01,
            4.8741e-01,  2.5108e-01],
          ...,
     

**Pregunta de análisis:**

Describí paso a paso cómo quedan alineadas las dimensiones después de reorganizar `medias`. Completá la tabla:

```
X: 32 3 28 28
medias: ? ? ? ?
```

medias: 1 3 1 1

### Ejercicio 10 — Cuadrícula con broadcasting bidireccional

**Objetivo:** Usar broadcasting en dos ejes al mismo tiempo para generar una matriz de combinaciones.

**Enunciado:**

Tenés un vector `x` con los números del 1 al 5 y un vector `y` con los múltiplos de 10 del 10 al 40.

Generá una matriz `M` de dimensiones `(5, 4)` donde cada posición `(i, j)` contenga la suma:
$$M_{i,j} = x_i + y_j$$

**Todo debe resolverse en una sola línea de código, sin bucles y sin matrices intermedias vacías.**

> **Pista:** Para que el broadcasting cree una matriz de sumas, uno de los vectores debe expandirse hacia abajo (columna) y el otro hacia la derecha (fila). Transformá `x` en un tensor columna de forma `(5, 1)` usando `.unsqueeze(1)`. El vector `y` de forma `(4,)` equivale automáticamente a una fila `(1, 4)`. Al sumarlos, PyTorch generará la cuadrícula completa.

In [84]:
import torch
x = torch.arange(1, 6) # [1, 2, 3, 4, 5]
y = torch.arange(10, 50, 10) # [10, 20, 30, 40]

# Tu código aquí: generá M en una sola línea
M = x.unsqueeze(1) + y.unsqueeze(0)
print(M)

tensor([[11, 21, 31, 41],
        [12, 22, 32, 42],
        [13, 23, 33, 43],
        [14, 24, 34, 44],
        [15, 25, 35, 45]])


---
## Sección D: Diferenciación automática con Autograd

### Introducción a Autograd

Una de las capacidades más importantes de PyTorch es su motor de **diferenciación automática**, llamado `autograd`. Cuando entrenamos una red neuronal, necesitamos calcular el gradiente de la función de pérdida con respecto a cada parámetro de la red. Hacer esto a mano para redes con millones de parámetros sería imposible.

PyTorch resuelve esto de forma elegante: a medida que realizás operaciones sobre tensores con `requires_grad=True`, PyTorch construye internamente un **grafo computacional** que registra qué operaciones se realizaron y en qué orden. Cuando llamás a `.backward()`, PyTorch recorre ese grafo al revés y calcula todos los gradientes automáticamente.

Después de llamar a `.backward()`, el grafo se libera de memoria por defecto (sólo se puede usar una vez).

### Ejercicio 11 — Gradiente automático y verificación analítica

**Objetivo:** Usar `autograd` para calcular una derivada y verificar el resultado con el cálculo analítico.

**Enunciado:**

1. Definí un tensor escalar `x = 2.0` e indicale a PyTorch que debe rastrear sus operaciones para calcular gradientes.
2. Calculá la función:
$$y = 3x^2 + 4x + 2$$
3. Ejecutá la propagación hacia atrás (`.backward()`) y extraé el valor del gradiente en `x`.
4. Verificá el resultado comparándolo con la derivada calculada analíticamente:
$$\frac{dy}{dx} = 6x + 4$$
evaluada en $x = 2$.

5. Intentá llamar a `.backward()` una segunda vez y observá el error. Luego volvé a calcular `y` usando `retain_graph=True` en el primer `.backward()` y verificá que ya no hay error.



In [89]:
import torch

# Paso 1 y 2: definir x e y
# Tu código aquí
x = torch.tensor(2.0, requires_grad=True)
y = 3*x**2 + 4*x + 2

# Paso 3: calcular el gradiente
# Tu código aquí
y.backward(retain_graph=True)
print(x.grad)

# Paso 4: verificar contra derivada analítica
# Tu código aquí
print(6*x + 4)
y.backward()


tensor(16.)
tensor(16., grad_fn=<AddBackward0>)


In [ ]:
# Paso 5: intentar backward() dos veces
# Tu código aquí


**Pregunta de análisis:**

1. ¿Qué es el **grafo computacional** que construye PyTorch? ¿Por qué se libera automáticamente después de llamar a `.backward()`?
2. ¿Para qué sirve `retain_graph=True`? ¿Investiga en qué situación real podría necesitarse calcularlo más de una vez?

¿Qué es el grafo computacional?
Es una estructura dinámica que PyTorch construye durante el forward, donde cada operación sobre tensores se guarda como un nodo conectado, permitiendo rastrear cómo se obtuvo cada valor y así calcular automáticamente los gradientes usando backpropagation.

¿Por qué se libera después de .backward()?
Porque el grafo ocupa memoria (guarda todas las operaciones intermedias), y una vez calculados los gradientes ya no es necesario; PyTorch lo elimina para ahorrar memoria y evitar consumo innecesario durante el entrenamiento.

¿Para qué sirve retain_graph=True y cuándo usarlo?
Sirve para evitar que el grafo se libere tras .backward(), permitiendo calcular gradientes más de una vez sobre el mismo forward; se usa, por ejemplo, cuando tenés múltiples pérdidas sobre la misma salida o en casos como GANs o cálculos de gradientes de orden superior.

---
## Sección E: Datasets personalizados con `torch.utils.data.Dataset`

Para entrenar redes con datos propios, PyTorch requiere que los encapsulés en una clase que herede de `torch.utils.data.Dataset`. Basta con implementar tres métodos:

| Método | Responsabilidad |
|---|---|
| `__init__` | Cargar metadatos, CSV, lista de rutas — lo que sea necesario para encontrar los datos |
| `__len__` | Devolver la cantidad total de elementos |
| `__getitem__(i)` | Cargar y retornar el elemento en la posición `i` |

Una vez que tu clase implementa esta interfaz, podés pasarla directamente a un `DataLoader` para obtener lotes, mezclado y carga paralela sin escribir código adicional.

In [90]:
# Descarga del dataset — ejecutar una sola vez
# Requiere: pip install gdown
!gdown https://drive.google.com/uc?id=1FMkstj2JgQOySU0D6mH7Dtr-hts2cqSD
!unzip -q plates.zip -d ./data/plates

Downloading...
From (original): https://drive.google.com/uc?id=1FMkstj2JgQOySU0D6mH7Dtr-hts2cqSD
From (redirected): https://drive.google.com/uc?id=1FMkstj2JgQOySU0D6mH7Dtr-hts2cqSD&confirm=t&uuid=5a9bb19a-f32f-4687-9c77-8ff7e1996de6
To: /content/plates.zip
100% 153M/153M [00:01<00:00, 118MB/s]
checkdir:  cannot create extraction directory: ./data/plates
           No such file or directory


### Ejercicio 12 — Creación de un Dataset personalizado

**Objetivo:** Implementar la interfaz `torch.utils.data.Dataset` para cargar un conjunto de datos propio desde archivos en disco.

**Contexto:** Trabajarás con un dataset de imágenes de placas de matrícula de distintos estados de EE.UU. El archivo `plates.csv` tiene la siguiente estructura:

| Columna | Descripción |
|---|---|
| `class id` | Identificador numérico del estado |
| `filepaths` | Ruta relativa a la imagen |
| `labels` | Nombre del estado (ej. `ALABAMA`) |
| `data set` | Partición: `train`, `test` o `val` |

**Enunciado:**

Completá la clase `PlatesDataSet` que hereda de `torch.utils.data.Dataset`:

1. `__init__`: leé el CSV con `pandas`, filtrá las filas según el valor de `mode` (`'train'`, `'test'` o `'val'`), y almacená las rutas, etiquetas numéricas y etiquetas de texto en atributos de la instancia.
2. `__len__`: retorná la cantidad de imágenes en la partición seleccionada.
3. `__getitem__(index)`: construí la ruta completa con `os.path.join()`, cargá la imagen con `PIL.Image.open()`, aplicá `transform` si fue provisto, y retorná el par `(imagen, etiqueta_numérica)`.
4. `get_class_name(index)`: retorná el nombre del estado (etiqueta de texto) para el elemento en `index`. Este método es necesario para la función de visualización.

> **Pistas:**
> - Usá `pd.read_csv()` para leer el CSV y filtrá con `df[df['data set'] == mode].reset_index(drop=True)`.
> - Recordá llamar a `super().__init__()` en el constructor.
> - En `__getitem__`, usá `.convert('RGB')` para garantizar 3 canales independientemente del formato de la imagen.

In [ ]:
import os
import pandas as pd
import torch
from torch.utils.data import Dataset
from PIL import Image
import torchvision.transforms as transforms


class PlatesDataSet(Dataset):
    def __init__(self, csv_file='./data/plates/plates.csv', root_dir='./data/plates',
                 mode='train', transform=None):
        """
        Parámetros:
        csv_file (str): ruta al archivo CSV con las anotaciones
        root_dir (str): directorio raíz que contiene las imágenes
        mode (str): partición a cargar ('train', 'test' o 'val')
        transform: transformaciones opcionales a aplicar a las imágenes
        """
        # Tu código aquí
        df = pd.read_csv(csv_file)
        df = df[df['data set'] == mode].reset_index(drop=True)


    def __len__(self):
        """Retorna la cantidad de imágenes en la partición."""
        # Tu código aquí

    def __getitem__(self, index):
        """
        Retorna la imagen y la etiqueta en la posición index.

        Retorna:
        tuple: (imagen, etiqueta_numérica)
        """
        # Tu código aquí

    def get_class_name(self, index):
        """Retorna el nombre del estado para el elemento en index."""
        # Tu código aquí

In [ ]:
# Función de visualización y test — ejecutar sin modificar
import matplotlib.pyplot as plt
import random
import numpy as np

def visualize_state_plates(dataset, state_name, num_examples=5):
    """
    Muestra ejemplos de placas de un estado específico del dataset.

    Parámetros:
    dataset (PlatesDataSet): dataset de placas de matrícula
    state_name (str): nombre del estado en mayúsculas (ej. 'NEW YORK')
    num_examples (int): cantidad de imágenes a mostrar
    """
    indices = [i for i in range(len(dataset))
               if dataset.get_class_name(i).upper() == state_name.upper()]

    if not indices:
        available = sorted(set(dataset.text_labels))
        print(f"No se encontraron imágenes para '{state_name}'.")
        print(f"Estados disponibles: {', '.join(available)}")
        return

    selected = random.sample(indices, min(num_examples, len(indices)))
    fig, axes = plt.subplots(1, len(selected), figsize=(len(selected) * 3, 3))
    if len(selected) == 1:
        axes = [axes]

    for i, idx in enumerate(selected):
        image, _ = dataset[idx]
        img_np = image.permute(1, 2, 0).numpy() if isinstance(image, torch.Tensor) else np.array(image)
        axes[i].imshow(img_np)
        axes[i].set_title(f"ID: {idx}")
        axes[i].axis('off')

    plt.suptitle(f"Placas — {state_name.upper()}", y=1.02)
    plt.tight_layout()
    plt.show()


# Probar el dataset con el estado 'NEW YORK'
dataset_train = PlatesDataSet(mode='train', transform=transforms.ToTensor())
print(f"Imágenes de entrenamiento: {len(dataset_train)}")
visualize_state_plates(dataset_train, 'NEW YORK')

**Pregunta de análisis:**

¿Por qué PyTorch separa la lógica del dataset (`Dataset`) de la lógica del iterador por lotes (`DataLoader`)? ¿Qué ventaja tiene este diseño a la hora de experimentar con distintos tamaños de lote o estrategias de mezclado de datos?

*(Escribí tu respuesta acá)*

---
## ¡Listo!

Completaste el Laboratorio 1a. En el próximo laboratorio usaremos todos estos conceptos como base para construir y entrenar redes neuronales reales sobre datos de imágenes.